# ClockInFace API — Webhook Integral Demo (attendance.created)

This notebook demonstrates a complete end-to-end integration:

1️⃣ Use a backend-issued API token  
2️⃣ Expose a public webhook endpoint via ngrok  
3️⃣ Subscribe to the attendance.created event  
4️⃣ Trigger a real facial recognition  
5️⃣ Receive webhook events in real time

---
## 1. Install dependencies

In [ ]:
!pip install requests flask pyngrok --quiet

---
## 2. API Configuration

In [ ]:
import os

BASE_URL = "https://backend.clockinface.com/api/v1"

# 🔐 Token provided securely by backend
ACCESS_TOKEN = os.getenv("CLOCKINFACE_TOKEN") or input("Enter API Token: ")

HEADERS = {
    "Authorization": f"Bearer {ACCESS_TOKEN}"
}

---
## 3. Configure ngrok

In [ ]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = os.getenv("NGROK_AUTH_TOKEN") or input("Enter ngrok auth token: ")
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

---
## 4. Create public webhook endpoint

In [ ]:
from flask import Flask, request
import threading

app = Flask(__name__)
event_count = 0

@app.route("/webhook", methods=["POST"])
def webhook():
    global event_count
    event_count += 1
    payload = request.json
    print(f"\n📩 Webhook received #{event_count}:")
    print(payload)
    return {"status": "ok"}, 200

def run():
    app.run(port=5000, use_reloader=False)

threading.Thread(target=run).start()

public_url = ngrok.connect(5000)
WEBHOOK_URL = public_url.public_url.replace("http://", "https://") + "/webhook"

print("🌍 Public webhook URL:")
print(WEBHOOK_URL)

---
## 5. Register webhook subscription

In [ ]:
import requests

payload = {
    "url": WEBHOOK_URL,
    "events": [1],
    "is_active": True
}

response = requests.post(
    f"{BASE_URL}/webhook/",
    json=payload,
    headers=HEADERS
)

if response.status_code != 201:
    raise Exception(f"Webhook creation failed: {response.text}")

webhook_info = response.json()
WEBHOOK_UUID = webhook_info["uuid"]
WEBHOOK_SECRET = webhook_info["secret"]

print("✅ Webhook registered")
print("Webhook UUID:", WEBHOOK_UUID)

---
## 6. Trigger real recognition (Integral Test)

### Register test person

In [ ]:
import json
from io import BytesIO

def image_from_url(url):
    r = requests.get(url)
    r.raise_for_status()
    return BytesIO(r.content)

characters = [
    (1, "Ben Affleck", "https://cdn.britannica.com/33/242333-050-95A19CE8/Actor-Ben-Affleck-premiere-AIR-March-2023.jpg?w=300"),
]

registered = []

for id, name, img_url in characters:
    img_file = image_from_url(img_url)

    payload = {
        "external_id": f"demo-{id}",
        "metadata": json.dumps([
            {"key": "id_card", "label": "ID Card", "value": str(id)},
            {"key": "name", "label": "Name", "value": name}
        ])
    }

    files = {
        "image": (f"{name.replace(' ', '_').lower()}.jpg", img_file, "image/jpeg")
    }

    r = requests.post(
        f"{BASE_URL}/person/register_person/",
        headers=HEADERS,
        data=payload,
        files=files,
    )

    registered.append(r.json())

registered

### Send recognition request

In [ ]:
RECOGNITION_TAGS = "entrance, webcam01"
METADATA = {"login_session_id": "abcdef123456"}

def recognize(img_url, tags=RECOGNITION_TAGS, metadata=METADATA):
    payload = {
        'tags': tags,
        'metadata': json.dumps(metadata),
    }

    img_file = image_from_url(img_url)

    files = {
        "image": ("unknown.jpg", img_file, "image/jpeg")
    }

    return requests.post(
        f"{BASE_URL}/person/facial_recognition/",
        headers=HEADERS,
        data=payload,
        files=files,
    ).json()

In [ ]:
img_url = "https://cdn.britannica.com/33/242333-050-95A19CE8/Actor-Ben-Affleck-premiere-AIR-March-2023.jpg?w=300"

result = recognize(img_url)
print("Recognition result:")
print(json.dumps(result, indent=2))

## 🔴 LIVE DEMO
If attendance automation is enabled, webhook events will appear above instantly.

---
## 7. Manage webhooks

In [ ]:
requests.get(f"{BASE_URL}/webhook/", headers=HEADERS).json()

In [ ]:
requests.patch(
    f"{BASE_URL}/webhook/{WEBHOOK_UUID}/",
    json={"is_active": False},
    headers=HEADERS
).json()

In [ ]:
requests.delete(
    f"{BASE_URL}/webhook/{WEBHOOK_UUID}/",
    headers=HEADERS
).status_code

---
## 8. Shutdown ngrok

In [ ]:
ngrok.kill()
print("🛑 ngrok tunnel closed")